# 01 — CNN Baseline Training & Evaluation

**Architecture:** Zhang et al. "Sequence-to-Point Learning with Neural Networks for NILM" (AAAI 2018)  
**Model:** 5× Conv1D → center slice → Dense(1024) → Multi-task heads  
**Parameters:** 101K | INT8: 0.101 MB  
**Dataset:** UK-DALE (primary) + REDD, AMPds2, REFIT (cross-dataset)  

**Sections:**
1. Environment Setup
2. Data Loading & Statistics
3. Model Architecture Visualization
4. Training (100 epochs)
5. Training Curves (Loss, MR, F1, MAE vs Epoch)
6. Final Evaluation Table (NILMFormer format)
7. Predictions vs Ground Truth (per appliance)
8. Confusion Matrix (per appliance)
9. Error Analysis (when does the model fail?)
10. Per-Appliance Bar Charts
11. Save Results to Drive

**Author:** Chadha Jeddi — NILM Benchmarking Project

---
## 1. Environment Setup
Load configuration from `00_setup.ipynb` (must be run first).

In [ ]:
# ---- Load Colab setup config ----
import json, sys, os

from google.colab import drive
drive.mount('/content/drive')

DRIVE_NILM = '/content/drive/MyDrive/nilm_project'
REPO_DIR = '/content/nilm-benchmarking'

# Clone repo if not already done
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}

os.chdir(REPO_DIR)

# Python paths
for p in ['src', 'models', 'models/baselines', 'models/proposed']:
    full = f'{REPO_DIR}/{p}'
    if full not in sys.path:
        sys.path.insert(0, full)

# Symlinks for data
for d in ['data/raw/UKDALE', 'data/raw/REDD', 'data/raw/AMPds2',
          'data/raw/REFIT', 'data/processed']:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

# Link data files from Drive
links = {
    'data/raw/UKDALE/ukdale.h5': f'{DRIVE_NILM}/data/raw/UKDALE/ukdale.h5',
    'data/raw/REDD/redd.h5': f'{DRIVE_NILM}/data/raw/REDD/redd.h5',
}
for local, remote in links.items():
    if os.path.exists(remote) and not os.path.exists(local):
        os.symlink(remote, local)

# Link experiments to Drive (persistent)
for folder in ['checkpoints', 'results']:
    src = f'{DRIVE_NILM}/experiments/{folder}'
    dst = f'{REPO_DIR}/experiments/{folder}'
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        import shutil; shutil.rmtree(dst)
    if not os.path.islink(dst):
        os.symlink(src, dst)

# Link processed cache to Drive
proc_src = f'{DRIVE_NILM}/data/processed'
proc_dst = f'{REPO_DIR}/data/processed'
os.makedirs(proc_src, exist_ok=True)
if os.path.exists(proc_dst) and not os.path.islink(proc_dst):
    import shutil; shutil.rmtree(proc_dst)
if not os.path.islink(proc_dst):
    os.symlink(proc_src, proc_dst)

# Install dependencies
!pip install -q PyWavelets pyarrow h5py tqdm einops omegaconf torchinfo seaborn

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY'}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ---- Import all project modules ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

from config import (
    WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES,
    APPLIANCE_NAMES, APPLIANCES, BATCH_SIZE, SEED
)
from preprocessing import load_ukdale_house, preprocess_house
from dataset import NILMDataset, NormStats, split_train_val, build_dataloaders
from metrics import MetricsTracker, multi_task_loss, focal_loss
from cnn_model import CNNBaseline

# Plot style
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('Set2')

COLORS = {
    'kettle': '#D94040', 'fridge': '#2E9E5A',
    'washing_machine': '#E8922A', 'dishwasher': '#7B4FBF',
    'microwave': '#CC3399',
}

# Constants
MODEL_NAME = 'cnn'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 100
LR = 1e-3
PATIENCE = 15

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Epochs: {EPOCHS}")
print(f"\n✅ All imports successful")

---
## 2. Data Loading & Statistics

In [ ]:
# ---- Load and preprocess UK-DALE House 1 ----
from dataset import load_clean_df, save_clean_df

cached = load_clean_df('UK-DALE', 1)
if cached is not None:
    clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

print(f"\nClean data: {clean_df.shape}")
print(f"Date range: {clean_df.index[0]} to {clean_df.index[-1]}")
print(f"Duration: {(clean_df.index[-1] - clean_df.index[0]).days} days")

In [ ]:
# ---- Dataset statistics table ----
stats_rows = []
for a in APPLIANCE_NAMES:
    power = clean_df[a]
    state = clean_df[f'{a}_state']
    stats_rows.append({
        'Appliance': a,
        'Threshold (W)': APPLIANCES[a]['power_threshold'],
        'Max Power (W)': APPLIANCES[a]['max_power'],
        'Mean (W)': f'{power.mean():.1f}',
        'Duty Cycle (%)': f'{state.mean()*100:.2f}',
        'ON Events': int((state.diff() == 1).sum()),
    })
pd.DataFrame(stats_rows)

In [ ]:
# ---- Build DataLoaders ----
train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
print(f"Train: {len(train_df):,} rows | Val: {len(val_df):,} rows")

train_loader, val_loader, norm_stats = build_dataloaders(
    train_df, val_df,
    batch_size=256,
    train_stride=1,
    val_stride=480,       # non-overlapping for faster validation
    num_workers=2,
    add_temporal_features=True,
)

# Verify shapes
x_sample, yp_sample, ys_sample = next(iter(val_loader))
print(f"\nBatch shapes:")
print(f"  x:       {tuple(x_sample.shape)}")
print(f"  y_power: {tuple(yp_sample.shape)}")
print(f"  y_state: {tuple(ys_sample.shape)}")

---
## 3. Model Architecture

In [ ]:
from torchinfo import summary

model = CNNBaseline(
    in_channels=INPUT_CHANNELS,
    window_size=WINDOW_SIZE,
    n_appliances=N_APPLIANCES,
    dropout=0.5,
).to(DEVICE)

print(f"Model: CNN Baseline (Zhang et al. AAAI 2018)")
print(f"Reference: https://doi.org/10.1609/aaai.v32i1.11873\n")

summary(model, input_size=(1, INPUT_CHANNELS, WINDOW_SIZE),
        col_names=['input_size', 'output_size', 'num_params'],
        depth=3)

---
## 4. Training Loop (100 epochs)

In [ ]:
from train import train_one_epoch, validate_one_epoch, EarlyStopping
import time

# ---- Optimizer and scheduler ----
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)
early_stop = EarlyStopping(patience=PATIENCE)

# ---- Metrics tracker ----
app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)

# ---- Training history ----
history = {
    'epoch': [], 'train_loss': [], 'val_loss': [],
    'val_mr': [], 'val_f1': [], 'val_mae': [], 'lr': [],
}

best_mr = -float('inf')
best_state = None
best_epoch = 0
start_time = time.time()

print(f"Training CNN for {EPOCHS} epochs on {DEVICE}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print('=' * 80)

for epoch in range(1, EPOCHS + 1):
    ep_start = time.time()

    # Train
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)

    # Validate
    val_loss, metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)

    val_mr = metrics['mean']['mr']
    val_f1 = metrics['mean']['f1']
    val_mae = metrics['mean']['mae_w']
    cur_lr = optimizer.param_groups[0]['lr']
    ep_time = time.time() - ep_start

    # Log
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mr'].append(val_mr)
    history['val_f1'].append(val_f1)
    history['val_mae'].append(val_mae)
    history['lr'].append(cur_lr)

    # Best model
    is_best = val_mr > best_mr
    if is_best:
        best_mr = val_mr
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Print
    star = ' ★' if is_best else ''
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
          f"MR: {val_mr:.3f} | F1: {val_f1:.3f} | "
          f"MAE: {val_mae:.1f}W | LR: {cur_lr:.1e} | "
          f"{ep_time:.0f}s{star}")

    # Scheduler
    scheduler.step(val_mr)

    # Early stopping
    if early_stop.step(val_mr):
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time/60:.1f} minutes")
print(f"Best epoch: {best_epoch} | Best MR: {best_mr:.4f}")

---
## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs = history['epoch']

# --- Loss ---
axes[0, 0].plot(epochs, history['train_loss'], label='Train', color='#3366CC')
axes[0, 0].plot(epochs, history['val_loss'], label='Validation', color='#D94040')
axes[0, 0].axvline(best_epoch, color='gray', linestyle=':', alpha=0.5, label=f'Best (ep {best_epoch})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Multi-task Loss (Focal + SmoothL1 + Gated)')
axes[0, 0].legend()

# --- MR ---
axes[0, 1].plot(epochs, history['val_mr'], color='#2E9E5A', linewidth=2)
axes[0, 1].axvline(best_epoch, color='gray', linestyle=':', alpha=0.5)
axes[0, 1].axhline(best_mr, color='#2E9E5A', linestyle='--', alpha=0.3, label=f'Best MR: {best_mr:.3f}')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Matching Ratio ↑')
axes[0, 1].set_title('Validation MR (checkpoint selection metric)')
axes[0, 1].legend()

# --- F1 ---
axes[1, 0].plot(epochs, history['val_f1'], color='#E8922A', linewidth=2)
axes[1, 0].axvline(best_epoch, color='gray', linestyle=':', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('F1 Score ↑')
axes[1, 0].set_title('Validation F1 Score')

# --- MAE ---
axes[1, 1].plot(epochs, history['val_mae'], color='#7B4FBF', linewidth=2)
axes[1, 1].axvline(best_epoch, color='gray', linestyle=':', alpha=0.5)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('MAE (W) ↓')
axes[1, 1].set_title('Validation Mean Absolute Error')

plt.suptitle(f'CNN Baseline — Training Curves (Best epoch: {best_epoch})', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: experiments/results/{MODEL_NAME}_training_curves.png")

---
## 6. Final Evaluation Table
Load best checkpoint and evaluate — matching NILMFormer Table 2 format.

In [ ]:
# ---- Load best model ----
model.load_state_dict(best_state)
model = model.to(DEVICE)
model.eval()

# ---- Final evaluation ----
_, final_metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)

print(f"\n{'='*80}")
print(f"FINAL RESULTS — CNN Baseline (epoch {best_epoch})")
print(f"{'='*80}")
tracker.print_table(final_metrics)

# ---- Save as DataFrame ----
results_df = tracker.to_dataframe(final_metrics)
results_df.to_csv(f'experiments/results/{MODEL_NAME}_metrics.csv', index=False)
print(f"\nSaved: experiments/results/{MODEL_NAME}_metrics.csv")

In [ ]:
# ---- Styled results table (thesis-ready) ----
display_df = results_df.copy()
display_df = display_df.set_index('appliance')
display_df.columns = [
    'MAE↓(W)', 'MSE', 'RMSE↓', 'NDE↓', 'SAE↓',
    'TECA↑', 'MR↑', 'F1↑', 'Acc↑', 'BAcc↑', 'Prec↑', 'Rec↑'
]
display_df = display_df.drop(columns=['MSE'])
display_df = display_df.round(3)
display_df.style.highlight_max(
    subset=['MR↑', 'F1↑', 'Acc↑', 'BAcc↑', 'Prec↑', 'Rec↑', 'TECA↑'],
    color='#d4edda'
).highlight_min(
    subset=['MAE↓(W)', 'RMSE↓', 'NDE↓', 'SAE↓'],
    color='#d4edda'
)

---
## 7. Predictions vs Ground Truth
Visual comparison for each appliance — the most informative plot for NILM evaluation.

In [ ]:
# ---- Collect predictions on validation set ----
all_pred_power = []
all_true_power = []
all_pred_state = []
all_true_state = []

model.eval()
with torch.no_grad():
    for x, yp, ys in val_loader:
        x = x.to(DEVICE)
        pp, ps, pg = model(x)
        all_pred_power.append(pp.cpu().numpy())
        all_true_power.append(yp.numpy())
        # Convert logits to binary
        pred_binary = (torch.sigmoid(ps) >= 0.5).float()
        all_pred_state.append(pred_binary.cpu().numpy())
        all_true_state.append(ys.numpy())

pred_power = np.concatenate(all_pred_power, axis=0)
true_power = np.concatenate(all_true_power, axis=0)
pred_state = np.concatenate(all_pred_state, axis=0)
true_state = np.concatenate(all_true_state, axis=0)

print(f"Predictions collected: {pred_power.shape[0]} samples")

In [ ]:
# ---- Predictions vs Ground Truth — per appliance ----
# Show 500 consecutive timesteps for visual clarity
N_SHOW = 500
start_idx = len(pred_power) // 3  # pick a window from the middle

fig, axes = plt.subplots(N_APPLIANCES, 1, figsize=(16, 3 * N_APPLIANCES), sharex=True)

for i, appliance in enumerate(APPLIANCE_NAMES):
    ax = axes[i]
    max_w = APPLIANCES[appliance]['max_power']

    true_w = true_power[start_idx:start_idx + N_SHOW, i] * max_w
    pred_w = pred_power[start_idx:start_idx + N_SHOW, i] * max_w

    ax.plot(true_w, color=COLORS[appliance], alpha=0.7, linewidth=1.2, label='Ground Truth')
    ax.plot(pred_w, color='#333333', alpha=0.6, linewidth=0.8, linestyle='--', label='Predicted')
    ax.fill_between(range(N_SHOW), true_w, pred_w, alpha=0.15, color='red', label='Error')
    ax.set_ylabel(f'{appliance}\n(W)', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)

    # Add MAE annotation
    mae_val = final_metrics[appliance]['mae_w']
    ax.text(0.02, 0.85, f'MAE={mae_val:.1f}W', transform=ax.transAxes,
            fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Timestep (validation set)')
plt.suptitle(f'CNN Baseline — Predictions vs Ground Truth ({N_SHOW} timesteps)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Confusion Matrix (per appliance)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(4 * N_APPLIANCES, 4))

for i, appliance in enumerate(APPLIANCE_NAMES):
    cm = confusion_matrix(true_state[:, i], pred_state[:, i], labels=[0, 1])
    # Normalize by row (true class)
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-8)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=['OFF', 'ON'], yticklabels=['OFF', 'ON'],
                ax=axes[i], cbar=False, vmin=0, vmax=1)
    axes[i].set_title(f'{appliance}\nF1={final_metrics[appliance]["f1"]:.3f}', fontsize=11)
    axes[i].set_xlabel('Predicted')
    if i == 0:
        axes[i].set_ylabel('Actual')
    else:
        axes[i].set_ylabel('')

plt.suptitle('CNN Baseline — Normalized Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Error Analysis
When does the model fail? Analyze errors by appliance duty cycle and overlap.

In [ ]:
# ---- Error analysis: False Positives vs False Negatives ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fp_rates = []
fn_rates = []
for i, a in enumerate(APPLIANCE_NAMES):
    ts, ps = true_state[:, i], pred_state[:, i]
    fp = ((ts == 0) & (ps == 1)).sum() / max((ts == 0).sum(), 1)
    fn = ((ts == 1) & (ps == 0)).sum() / max((ts == 1).sum(), 1)
    fp_rates.append(fp)
    fn_rates.append(fn)

x_pos = np.arange(N_APPLIANCES)
width = 0.35

axes[0].bar(x_pos - width/2, fp_rates, width, label='False Positive Rate',
            color='#E8922A', alpha=0.8)
axes[0].bar(x_pos + width/2, fn_rates, width, label='False Negative Rate',
            color='#3366CC', alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(APPLIANCE_NAMES, rotation=20)
axes[0].set_ylabel('Rate')
axes[0].set_title('Error Types per Appliance')
axes[0].legend()

# ---- Power error distribution ----
for i, a in enumerate(APPLIANCE_NAMES):
    max_w = APPLIANCES[a]['max_power']
    errors = (pred_power[:, i] - true_power[:, i]) * max_w
    axes[1].hist(errors, bins=50, alpha=0.5, label=a, color=COLORS[a])

axes[1].set_xlabel('Power Error (W)')
axes[1].set_ylabel('Count')
axes[1].set_title('Power Error Distribution (centered at 0 = perfect)')
axes[1].legend(fontsize=8)
axes[1].set_xlim(-200, 200)

plt.suptitle('CNN Baseline — Error Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Per-Appliance Bar Charts
Visual summary for thesis figures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

appliances = APPLIANCE_NAMES
colors = [COLORS[a] for a in appliances]
x_pos = np.arange(len(appliances))

# --- F1 Score ---
f1_vals = [final_metrics[a]['f1'] for a in appliances]
bars = axes[0].bar(x_pos, f1_vals, color=colors, alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(appliances, rotation=20)
axes[0].set_ylabel('F1 Score ↑')
axes[0].set_title('F1 Score per Appliance')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars, f1_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', fontsize=9)

# --- MAE (W) ---
mae_vals = [final_metrics[a]['mae_w'] for a in appliances]
bars = axes[1].bar(x_pos, mae_vals, color=colors, alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(appliances, rotation=20)
axes[1].set_ylabel('MAE (W) ↓')
axes[1].set_title('Mean Absolute Error per Appliance')
for bar, val in zip(bars, mae_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}', ha='center', fontsize=9)

# --- MR ---
mr_vals = [final_metrics[a]['mr'] for a in appliances]
bars = axes[2].bar(x_pos, mr_vals, color=colors, alpha=0.85)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(appliances, rotation=20)
axes[2].set_ylabel('Matching Ratio ↑')
axes[2].set_title('Matching Ratio per Appliance')
axes[2].set_ylim(0, 1)
for bar, val in zip(bars, mr_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', fontsize=9)

plt.suptitle(f'CNN Baseline — Per-Appliance Results (epoch {best_epoch})', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Save Results to Drive
Everything persists on Google Drive for comparison notebook and thesis.

In [ ]:
import json

# ---- Save checkpoint ----
checkpoint = {
    'model_name': MODEL_NAME,
    'model_state_dict': best_state,
    'best_epoch': best_epoch,
    'best_val_mr': best_mr,
    'n_params': sum(p.numel() for p in model.parameters()),
    'norm_stats': {
        'agg_mean': norm_stats.agg_mean,
        'agg_std': norm_stats.agg_std,
        'appliance_max': norm_stats.appliance_max,
    },
}
ckpt_path = f'experiments/checkpoints/{MODEL_NAME}_best.pth'
torch.save(checkpoint, ckpt_path)
print(f"✅ Checkpoint saved: {ckpt_path}")

# ---- Save training history ----
history['training_time_seconds'] = total_time
history['best_epoch'] = best_epoch
history['best_val_mr'] = best_mr
history['model'] = MODEL_NAME

hist_path = f'experiments/results/{MODEL_NAME}_history.json'
with open(hist_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"✅ History saved: {hist_path}")

# ---- Save final metrics ----
tracker.save_json(
    f'experiments/results/{MODEL_NAME}_metrics.json',
    final_metrics,
    model_name=MODEL_NAME,
)

# ---- Save NormStats ----
norm_stats.save(f'experiments/results/{MODEL_NAME}_norm_stats.json')

print(f"\n{'='*60}")
print(f"All results saved to Google Drive (persistent!)")
print(f"{'='*60}")
print(f"\nFiles on Drive:")
print(f"  experiments/checkpoints/{MODEL_NAME}_best.pth")
print(f"  experiments/results/{MODEL_NAME}_history.json")
print(f"  experiments/results/{MODEL_NAME}_metrics.json")
print(f"  experiments/results/{MODEL_NAME}_metrics.csv")
print(f"  experiments/results/{MODEL_NAME}_norm_stats.json")
print(f"  experiments/results/{MODEL_NAME}_training_curves.png")
print(f"  experiments/results/{MODEL_NAME}_predictions.png")
print(f"  experiments/results/{MODEL_NAME}_confusion.png")
print(f"  experiments/results/{MODEL_NAME}_error_analysis.png")
print(f"  experiments/results/{MODEL_NAME}_bar_charts.png")

---
## Summary

| Item | Value |
|------|-------|
| Model | CNN Baseline (Zhang et al. AAAI 2018) |
| Parameters | 101K |
| INT8 Size | 0.101 MB |
| Training epochs | see above |
| Best MR | see above |
| Training time | see above |
| Dataset | UK-DALE House 1 |

**Next:** Run `02_train_gru.ipynb` — change `MODEL_NAME` and model class only.